# Summarization: Extractive Scoring + Mocked Map-Reduce

This notebook implements two summarization techniques from `../04-summarization-techniques.md`,
applied to a synthetic multi-paragraph "project history" -- mirroring the Virtual Liaison's "concise
project tracking & summarization" facility.

1. A **from-scratch extractive summarizer** that scores sentences by TF-IDF term weight and selects
   the top-N most "important" sentences.
2. A **mocked map-reduce summarization chain**: split the project history into chunks, "summarize"
   each chunk independently (mocked -- no real LLM call), then combine the partial summaries into one
   final summary.

Fully offline -- only `numpy`, `pandas`, and `scikit-learn` are used, no API keys.

In [1]:
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

pd.set_option("display.max_colwidth", 100)
print("Ready.")

Ready.


## 1. Synthetic multi-paragraph project history

A long-running project accumulates status updates over time. This is the kind of text that would
eventually exceed a single LLM context window on a real, months-long project.

In [2]:
project_history = """
Project Atlas kicked off in January with a scope covering Japanese and French localization of the
Q2 promotional campaign for a new oncology therapy. The initial project plan estimated a six week
timeline with translation review as the primary risk area given the technical medical terminology.

By early February the Japanese localization track was progressing smoothly. The translation vendor
delivered the first draft ahead of schedule and medical review flagged only minor terminology
corrections. The French localization track, however, hit a snag: legal review identified that the
updated promotional claims language needed additional compliance sign-off before translation could
proceed, introducing a two week delay to that track specifically.

Through March the Japanese track completed final review and was approved for launch, while the
French track remained blocked pending legal sign-off. The project team escalated the French delay to
the account lead, who negotiated an expedited compliance review with the client's legal team.

By early April the French legal review was resolved and translation resumed, catching up roughly one
week of the original two week delay by running translation and final formatting in parallel rather
than sequentially. The Japanese assets launched successfully in early April as planned.

As of mid April, Project Atlas is effectively back on track: the Japanese localization launched on
schedule, and the French localization is now projected to launch only about one week behind the
original six week estimate, following resolution of the legal review delay that affected that track
specifically in February and March.
""".strip()

paragraphs = [p.strip() for p in project_history.split("\n\n") if p.strip()]
print(f"{len(paragraphs)} paragraphs, {len(project_history.split())} words total.")

5 paragraphs, 248 words total.


## 2. Extractive summarization: sentence scoring by TF-IDF term weight

Split the full history into sentences, score each by the sum of its TF-IDF weights (a simple proxy
for "how much distinctive information this sentence carries"), and keep the top-N highest-scoring
sentences, in their original order, as a rough extractive summary.

In [3]:
def split_sentences(text: str) -> list[str]:
    # simple sentence splitter -- good enough for this synthetic, well-punctuated text
    text = text.replace("\n", " ")
    sentences = re.split(r"(?<=[.!?])\s+", text)
    return [s.strip() for s in sentences if s.strip()]


def extractive_top_sentences(sentences: list[str], top_n: int = 5) -> list[str]:
    vectorizer = TfidfVectorizer(stop_words="english")
    tfidf = vectorizer.fit_transform(sentences)
    scores = np.asarray(tfidf.sum(axis=1)).ravel()   # simple importance proxy
    top_idx = np.argsort(scores)[::-1][:top_n]
    return [sentences[i] for i in sorted(top_idx)]    # keep original chronological order


sentences = split_sentences(project_history)
print(f"{len(sentences)} sentences total.\n")

extractive_summary = extractive_top_sentences(sentences, top_n=5)
print("EXTRACTIVE SUMMARY (top 5 sentences by TF-IDF weight, chronological order):\n")
for s in extractive_summary:
    print("-", s)

10 sentences total.

EXTRACTIVE SUMMARY (top 5 sentences by TF-IDF weight, chronological order):

- Project Atlas kicked off in January with a scope covering Japanese and French localization of the Q2 promotional campaign for a new oncology therapy.
- The initial project plan estimated a six week timeline with translation review as the primary risk area given the technical medical terminology.
- The French localization track, however, hit a snag: legal review identified that the updated promotional claims language needed additional compliance sign-off before translation could proceed, introducing a two week delay to that track specifically.
- By early April the French legal review was resolved and translation resumed, catching up roughly one week of the original two week delay by running translation and final formatting in parallel rather than sequentially.
- As of mid April, Project Atlas is effectively back on track: the Japanese localization launched on schedule, and the French loca

Notice the extractive summary reads a bit disjointed -- it's five sentences pulled from different
points in the history, not a flowing narrative. That's the expected trade-off from Chapter 4:
extractive summarization is fast, cheap, and hallucination-free, but not naturally coherent.

## 3. Mocked map-reduce summarization chain

Now the abstractive approach: chunk the history, "summarize" each chunk independently (map), then
combine the partial summaries into one final summary (reduce). Both the map and reduce steps use a
**mock LLM function** -- deterministic, offline, no API key -- standing in for a real
`prompt | llm | StrOutputParser()` LCEL chain (see Chapter 4 and Chapter 1).

In [4]:
def mock_llm_summarize(text: str, max_sentences: int = 1) -> str:
    """Stand-in for a real LLM summarization call, e.g.:

        from langchain_core.prompts import ChatPromptTemplate
        summarize_chain = summarize_prompt | llm | StrOutputParser()
        summarize_chain.invoke({"text": text})

    Here we simulate 'abstractive-ish' behavior deterministically: extract the single most
    TF-IDF-important sentence from the chunk (reusing the extractive scorer) as a proxy for what an
    LLM would distill the chunk down to. No network call, no API key.
    """
    chunk_sentences = split_sentences(text)
    if not chunk_sentences:
        return ""
    top = extractive_top_sentences(chunk_sentences, top_n=max_sentences)
    return " ".join(top)


def map_reduce_summarize(chunks: list[str]) -> tuple[list[str], str]:
    # Map: summarize each chunk independently (in production, these calls can run concurrently)
    partial_summaries = [mock_llm_summarize(chunk, max_sentences=1) for chunk in chunks]
    # Reduce: combine the partial summaries into one final summary
    combined_text = " ".join(partial_summaries)
    final_summary = mock_llm_summarize(combined_text, max_sentences=3)
    return partial_summaries, final_summary


# Chunk by paragraph, mirroring "one chunk per status update" from Chapter 1's chunking guidance
chunks = paragraphs
partial_summaries, final_summary = map_reduce_summarize(chunks)

print("PER-CHUNK (MAP) SUMMARIES:\n")
for i, s in enumerate(partial_summaries, 1):
    print(f"[chunk {i}] {s}\n")

print("FINAL (REDUCE) SUMMARY:\n")
print(final_summary)

PER-CHUNK (MAP) SUMMARIES:

[chunk 1] The initial project plan estimated a six week timeline with translation review as the primary risk area given the technical medical terminology.

[chunk 2] The French localization track, however, hit a snag: legal review identified that the updated promotional claims language needed additional compliance sign-off before translation could proceed, introducing a two week delay to that track specifically.

[chunk 3] Through March the Japanese track completed final review and was approved for launch, while the French track remained blocked pending legal sign-off.

[chunk 4] By early April the French legal review was resolved and translation resumed, catching up roughly one week of the original two week delay by running translation and final formatting in parallel rather than sequentially.

[chunk 5] As of mid April, Project Atlas is effectively back on track: the Japanese localization launched on schedule, and the French localization is now projected t

## 4. Refine-style summarization (sequential, for comparison)

Chapter 4 also covers **refine** summarization -- sequentially updating a running summary chunk by
chunk, which preserves narrative continuity better than map-reduce at the cost of being strictly
sequential. Here's the control-flow shape (still mocked, no real LLM call):

In [5]:
def mock_llm_refine(previous_summary: str, new_chunk: str) -> str:
    """Stand-in for a refine-prompt LLM call: 'here is the summary so far, here is the next
    update, produce an updated summary.' We simulate this by re-scoring the combined text and
    keeping the top sentences, so later chunks can 'pull forward' earlier context.
    """
    combined = (previous_summary + " " + new_chunk).strip()
    return mock_llm_summarize(combined, max_sentences=2)


def refine_summarize(chunks: list[str]) -> str:
    summary = mock_llm_refine("", chunks[0])
    for chunk in chunks[1:]:
        summary = mock_llm_refine(summary, chunk)
    return summary


refine_result = refine_summarize(chunks)
print("REFINE SUMMARY (sequential):\n")
print(refine_result)

REFINE SUMMARY (sequential):

The French localization track, however, hit a snag: legal review identified that the updated promotional claims language needed additional compliance sign-off before translation could proceed, introducing a two week delay to that track specifically. As of mid April, Project Atlas is effectively back on track: the Japanese localization launched on schedule, and the French localization is now projected to launch only about one week behind the original six week estimate, following resolution of the legal review delay that affected that track specifically in February and March.


## 5. Comparing the three outputs

All three techniques were applied to the *same* synthetic project history. Compare them side by side
-- this is exactly the kind of comparison worth being able to reproduce and narrate in an interview.

In [6]:
comparison = pd.DataFrame({
    "technique": ["Extractive (top-5 sentences)", "Map-reduce (mocked)", "Refine (mocked, sequential)"],
    "output": [
        " ".join(extractive_summary),
        final_summary,
        refine_result,
    ],
})
for row in comparison.itertuples():
    print(f"--- {row.technique} ---")
    print(row.output)
    print()

--- Extractive (top-5 sentences) ---
Project Atlas kicked off in January with a scope covering Japanese and French localization of the Q2 promotional campaign for a new oncology therapy. The initial project plan estimated a six week timeline with translation review as the primary risk area given the technical medical terminology. The French localization track, however, hit a snag: legal review identified that the updated promotional claims language needed additional compliance sign-off before translation could proceed, introducing a two week delay to that track specifically. By early April the French legal review was resolved and translation resumed, catching up roughly one week of the original two week delay by running translation and final formatting in parallel rather than sequentially. As of mid April, Project Atlas is effectively back on track: the Japanese localization launched on schedule, and the French localization is now projected to launch only about one week behind the orig

## Takeaways

- Extractive summarization is fast and never hallucinates, but reads as disjointed quoted fragments
  rather than a narrative -- fine as a pre-filter, not ideal as the final client-facing answer.
- Map-reduce and refine both produce a more digestible summary here (via the mocked abstractive
  step), but a real LLM-backed refine chain would better preserve the *causal* thread across chunks
  (e.g., "French was delayed... and later caught up") than map-reduce, which summarizes each chunk in
  isolation before combining -- see `../04-summarization-techniques.md` for the full trade-off
  discussion, including why refine was the more defensible choice for project tracking specifically
  and how the LangChain equivalents (`prompt | llm | StrOutputParser()`) would be wired in place of
  `mock_llm_summarize` / `mock_llm_refine` in production.